In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Introduction to Gemini 3

| Adapted from **Intro to Gemini** by:|
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Gemini 3 is Google's latest flagship model family, trained to be especially proficient in:

* Advanced reasoning and complex instruction following  
* Agentic operations and autonomous code execution  
* Multimodal understanding across long contexts (text, image, audio, video)

This notebook serves as a quickstart guide for developers to begin interacting with the **Gemini 3.1 Pro** and **Gemini 3 Flash** models via the Google Gen AI SDK on Vertex AI. It is designed to demonstrate key model capabilities and showcase the API features.

## Get started

In this section, you install and prepare your environment. When instructed, return to the lab instructions to check your progress and confirm your progress.

### Install Google Gen AI SDK for Python

Gemini 3 API features and Flash require Gen AI SDK for Python version 1.51.0 or later.

In [1]:
%pip install --upgrade --quiet google-genai>=1.56.0

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### Import libraries


In [ ]:
import os
import sys

from IPython.display import HTML, Markdown, display, Image
from google import genai
from google.genai import types
from pydantic import BaseModel

### Authenticate your Google Cloud Project for Vertex AI

You can use a Google Cloud Project or an API Key for authentication. This lab uses your Google Cloud Project.

- Enable the Vertex AI API by running the code below.

In [ ]:
import os
PROJECT_ID = "qwiklabs-gcp-02-f9d45a8d8912"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

Return to the lab instructions to **Check your progress**. This confirms you've installed the required packages and imported the required libraries.

## Load the Gemini models

You use the `gemini-3.1-pro-preview` and `gemini-3-flash-preview` models in this lab. Learn more about all [Gemini models on Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models#gemini-models).

In [ ]:
# Define both model IDs for use throughout the lab
PRO_MODEL_ID = "gemini-3.1-pro-preview"
FLASH_MODEL_ID = "gemini-3-flash-preview"

# Default to Pro for the main walkthrough
MODEL_ID = PRO_MODEL_ID

Return to the lab instructions to **Check your progress** to verify you've successfully set the chosen models.

## Explore Gemini 3.1 Pro

In this section, you explore key Gemini 3.1 Pro model capabilities and API features. When instructed, return to the lab instructions to check your progress and confirm your progress.

### ✅ Set system instructions

You can guide the behavior of Gemini models with system instructions. To do so, pass a `GenerateContentConfig` object.

In [ ]:
system_instruction = """
  You are a helpful language translator.
  Your mission is to translate text in English to Spanish.
"""

prompt = """
  User input: I like bagels.
  Answer:
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
    ),
)

display(Markdown(response.text))

Return to the lab instructions to **Check your progress** to verify you ran the **Set the system instructions** section.

### ✅ Thinking level

The `thinking_level` parameter allows you to control the cost of the model's response generation. By selecting a state, you can explicitly balance the trade-offs between response quality/reasoning complexity and latency/cost.

- **Minimum** (Flash only): Uses as few tokens as possible. Requires thought signatures; best used for low-complexity tasks that wouldn't benefit from extensive reasoning.
- **Low**: Minimizes latency and cost. Best for simple instruction following or chat.
- **Medium** (Flash only): Suitable for tasks of moderate complexity that benefit from reasoning but don't require deep, multi-step planning.
- **High**: Maximizes reasoning depth. The model may take significantly longer to reach a first token, but the output will be more thoroughly vetted.


#### ⚠️ Notes

- If `thinking_level` is not specified, the model defaults to `high`, which is a dynamic setting that adjusts based on prompt complexity.
- You cannot use both `thinking_level` and the legacy `thinking_budget` parameter in the same request. Doing so will return a 400 error.
- The OpenAI Chat Completions API `reasoning_effort` `medium` maps to `thinking_level` `high`.
- Thinking can't be turned off for Gemini 3.1 Pro.

In [ ]:
prompt = """
You are tasked with implementing the classic Thread-Safe Double-Checked Locking (DCL) Singleton pattern in modern C++. This task is non-trivial and requires specialized concurrency knowledge to prevent memory reordering issues.

Write a complete, runnable C++ program named `dcl_singleton.cpp` that defines a class `Singleton` with a private constructor and a static `getInstance()` method.

Your solution MUST adhere to the following strict constraints:
1. The Singleton instance pointer (`static Singleton*`) must be wrapped in `std::atomic` to correctly manage memory visibility across threads.
2. The `getInstance()` method must use `std::memory_order_acquire` when reading the instance pointer in the outer check.
3. The instance creation and write-back must use `std::memory_order_release` when writing to the atomic pointer.
4. A standard `std::mutex` must be used only to protect the critical section (the actual instantiation).
5. The `main` function must demonstrate safe, concurrent access by launching at least three threads, each calling `Singleton::getInstance()`, and printing the address of the returned instance to prove all threads received the same object.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH  # Dynamic thinking for high reasoning tasks
        )
    ),
)

display(Markdown(response.text))

For faster, lower-latency responses when complex reasoning isn't required, you can constrain the model's thinking level by setting parameter `thinking_level` to `low`:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="How does AI work?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.LOW  # For faster and lower-latency responses
        )
    ),
)

display(Markdown(response.text))

Gemini 3 Flash introduces a **MINIMAL** thinking level, which constraints the model to a near-zero reasoning budget. This is ideal for low-latency tasks.

In [ ]:
# --- GEMINI 3 FLASH COMPARISON: SPEED ---
# Switch to Flash using MINIMAL thinking for maximum speed

print(f"Generating with {FLASH_MODEL_ID} (Thinking Level: MINIMAL)...")

response = client.models.generate_content(
    model=FLASH_MODEL_ID,
    contents="Explain quantum entanglement in one sentence.",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.MINIMAL
        )
    ),
)

display(Markdown(response.text))

Return to the lab instruction to **Check your progress** to verify you ran **The thinking level** section.

### ✅ Configure the temperature parameter

⚠️ **Notes**: For Gemini 3, it's strongly recommended to keep the `temperature` parameter at its default value of `1.0`.

While previous models often benefited from tuning `temperature` to control creativity versus determinism, Gemini 3's reasoning capabilities are optimized for the default setting.


**Note:** Try changing the `temperature` value to `2.0` in the code block below to see how it affects the creativity of the response.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Tell me how the internet works, but pretend I'm a puppy who only understands squeaky toys.",
    config=types.GenerateContentConfig(
        temperature=1.0,
        top_p=0.9,
        max_output_tokens=8000,
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.LOW,
            include_thoughts=True,
        ),
    ),
)

display(Markdown(response.text))

### ✅ Generate content stream

The Gemini 3 content generation stream is a comprehensive, agentic ecosystem. It turns an idea or prompt into functional, visual, and interactive output, shifting from answering questions to building solutions.

In [ ]:
prompt = """
A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball.
How much does the ball cost?
"""

for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.LOW,
        )
    ),
):
    print(chunk.text, end="")

### ✅ Thought summaries

Thought summaries are summarized versions of the model's raw thoughts and offer insights into the model's internal reasoning process. Note that thinking levels and budgets apply to the model's raw thoughts and not to thought summaries.

You can include thought summaries in model response by setting `include_thoughts` to `true` as shown below:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="How many R's are in the word strawberry?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            include_thoughts=True,
            thinking_level=types.ThinkingLevel.LOW,
        )
    ),
)

for part in response.candidates[0].content.parts:
    if part.thought:
        display(
            Markdown(
                f"""## Thoughts:
         {part.text}
        """
            )
        )
    else:
        display(
            Markdown(
                f"""## Answer:
         {part.text}
        """
            )
        )

Return to the lab instructions to **Check your progress** to verify you ran the Thought summaries section.

### ✅  Multi-turn chat

Multi-turn chat collects multiple rounds of prompts and responses into a chat, which gives you an easy way to keep track of the conversation history.

The following is an example of starting and maintaining a multi-turn chat history.

In [ ]:
chat = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(),
)

In [ ]:
response = chat.send_message("Write a function that checks if a year is a leap year.")

display(Markdown(response.text))

This follow-up prompt shows how the model responds based on the previous prompt:

In [ ]:
response = chat.send_message("Write a unit test of the generated function.")

display(Markdown(response.text))

### ✅ Safety filters

The Gemini API's adjustable safety filters cover the following categories:
* Harassment: Negative or harmful comments targeting identity and/or protected attributes.
* Hate speech: Content that is rude, disrespectful, or profane.
* Sexually explicit: Contains references to sexual acts or other lewd content.
* Dangerous: Promotes, facilitates, or encourages harmful acts.
* Jailbreak: Targets *jailbreak* attempts. These are user prompts that try to manipulate the AI to ignore its safety training or system instructions.

In [ ]:
system_instruction = "Be as mean and hateful as possible."

prompt = """
Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark.
"""

safety_settings = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    # New category available for Gemini 3
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_JAILBREAK,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        safety_settings=safety_settings,
    ),
)

# Response will be `None` if it is blocked.
display(Markdown(response.text))
# Finish Reason will be `SAFETY` if it is blocked.
print(response.candidates[0].finish_reason)
# Safety Ratings show the levels for each filter.
for safety_rating in response.candidates[0].safety_ratings:
    print(safety_rating)

Return to the lab instructions to **Check your progress** to verify you ran the Safety filters section.

### ✅ Send asynchronous requests

Asynchronous requests to the Google Gemini Pro API enable non-blocking , parallel processing of multiple prompts. This is suitable for high-throughput applications and reduced total execution time.

In [ ]:
response = await client.aio.models.generate_content(
    model=MODEL_ID,
    contents="Compose a song about the adventures of a time-traveling squirrel.",
)

display(Markdown(response.text))

### ✅ Multimodality

- If your content is stored in [Google Cloud Storage](https://cloud.google.com/storage), you can use the `from_uri`  method to create a `Part` object.
- If your content is stored in your local file system, you can read it in as bytes data and use the `from_bytes` method to create a `Part` object.


#### 💡 **Image**

Gemini 3.1 Pro has native multimodal vision. This allows engineers to use images as high-dimensional inputs within the model's 1-million-token context window. It performs visual-spatial reasoning. It understands the relationships between components in complex diagrams, schematics, and UI mockups.

In this example, you use an image stored locally.

In [ ]:
# Download and open an image locally.
! wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/meal.png

with open("meal.png", "rb") as f:
    image = f.read()

# Display the image inline for the student to see
display(Image(data=image, width=400))

# Prompt the model with the image and text
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(data=image, mime_type="image/png"),
        "Write a short and engaging blog post based on this picture.",
    ],
)

display(Markdown(response.text))

#### 💡 **PDF**

Gemini 3.1 Pro offers native multimodal PDF understanding. It treats documents as primary visual and structural inputs instead of simple text. It processes up to 3,500 pages in its 1-million-token context window. This allows you to reason across complex layouts, diagrams, and massive technical specifications.

In this example, you use a PDF document stored on Google Cloud Storage.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/pdf/1706.03762v7.pdf",
            mime_type="application/pdf",
        ),
        "Summarize the document.",
    ],
)

display(Markdown(response.text))

#### 💡 **Audio**

Gemini 3.1 Pro offers native multimodal audio understanding. This allows engineers to process complex acoustic data without needing separate transcription services. It treats audio as a primary input, enabling synchronized reasoning across text, code, and sound. 

In this example, you use an audio file stored at a general web URL.


In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri="https://traffic.libsyn.com/secure/e780d51f-f115-44a6-8252-aed9216bb521/KPOD242.mp3",
            mime_type="audio/mpeg",
        ),
        "Write a summary of this podcast episode.",
    ],
    config=types.GenerateContentConfig(
        audio_timestamp=True,
    ),
)

display(Markdown(response.text))

#### 💡 **YouTube video**

The Gemini 3.1 Pro model has native multimodal video understanding. This lets you use video files or YouTube URLs as context for the model's reasoning engine. Gemini 3.1 Pro processes the visual frames and audio track at the same time.


In this example, you use this YouTube video [Google — 25 Years in Search: The Most Searched](https://www.youtube.com/watch?v=3KtWfp0UopM).


In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri="https://www.youtube.com/watch?v=3KtWfp0UopM",
            mime_type="video/mp4",
        ),
        "At what point in the video is Harry Potter shown?",
    ],
)

display(Markdown(response.text))

#### 💡 **Web Page (HTTP support)**

Gemini 3.1 Pro moves from analyzing content to automating and generating web content. It supports web page (HTTP) interaction with "agentic" and "vibe coding" features. These features allow it to read, analyze, and act upon live websites. 

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri="https://cloud.google.com/vertex-ai/generative-ai/docs",
            mime_type="text/html",
        ),
        "Write a summary of this documentation.",
    ],
)

display(Markdown(response.text))

### ✅ Media resolution

Gemini 3 introduces granular control over multimodal vision processing via the `media_resolution` parameter. Higher resolutions improve the model's ability to read fine text or identify small details, but increase token usage and latency. The `media_resolution` parameter determines the  maximum number of tokens allocated per input image or video frame.

**⚠️ Note**: Because media resolution directly impacts token count, you may need to lower the resolution (e.g., to `low`) to fit very long inputs, such as long videos or extensive documents.

You can set the resolution to `low`, `medium`, `high`, or, if using Flash, `ULTRA_HIGH` per individual media part:

In [ ]:
# Download image for preview
! gcloud storage cp gs://cloud-samples-data/generative-ai/image/a-man-and-a-dog.png .

# Display the image with a defined width
display(Image('a-man-and-a-dog.png', width=400))

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="gs://cloud-samples-data/generative-ai/image/a-man-and-a-dog.png",
                mime_type="image/jpeg",
            ),
            media_resolution=types.PartMediaResolution(
                level=types.PartMediaResolutionLevel.MEDIA_RESOLUTION_HIGH
            ),
        ),
        types.Part(
            file_data=types.FileData(
                file_uri="gs://cloud-samples-data/generative-ai/video/behind_the_scenes_pixel.mp4",
                mime_type="video/mp4",
            ),
            media_resolution=types.PartMediaResolution(
                level=types.PartMediaResolutionLevel.MEDIA_RESOLUTION_LOW
            ),
        ),
        "When does the image appear in the video? What is the context?",
    ],
)

display(Markdown(response.text))


Gemini 3 Flash supports an exclusive `ULTRA_HIGH` media resolution level for tasks requiring extreme visual detail.

In [ ]:
# --- GEMINI 3 FLASH COMPARISON: ULTRA HIGH VISION ---
# Use Flash with ULTRA_HIGH resolution for fine-grained details

print(f"Analyzing image with {FLASH_MODEL_ID} (Media Resolution: ULTRA_HIGH)...")

response = client.models.generate_content(
    model=FLASH_MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="gs://cloud-samples-data/generative-ai/image/a-man-and-a-dog.png",
                mime_type="image/jpeg",
            ),
            media_resolution=types.PartMediaResolution(
                level=types.PartMediaResolutionLevel.MEDIA_RESOLUTION_ULTRA_HIGH
            ),
        ),
        "Describe the texture of the dog's fur in extreme detail.",
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=types.ThinkingLevel.LOW)
    ),
)

display(Markdown(response.text))


Or set it globally via `GenerateContentConfig`. If unspecified, the model uses optimal defaults based on the media type.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="gs://cloud-samples-data/generative-ai/image/a-man-and-a-dog.png",
                mime_type="image/jpeg",
            ),
        ),
        "What is in the image?",
    ],
    config=types.GenerateContentConfig(
        media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW
    ),
)

display(Markdown(response.text))

### ✅ Structured output

Structured output in Gemini 3, including the Pro and Flash models, ensures that responses align with a predefined format. This results in reliable, type-safe, and machine-readable data for various applications. 

#### 💡 [Pydantic](https://docs.pydantic.dev/latest/) Model Schema support

The Gemini 3.1 Pro model has deep integration with Pydantic, a Python library for data validation. This lets engineers use Python type hints to define the structured output and agent schemas the model follows. This ensures predictable, type-safe data for production systems

In [ ]:
class CountryInfo(BaseModel):
    name: str
    population: int
    capital: str
    continent: str
    gdp: int
    official_language: str
    total_area_sq_mi: int


response = client.models.generate_content(
    model=MODEL_ID,
    contents="Give me information for the United States.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CountryInfo,
    ),
)
# Response as JSON
display(Markdown(response.text))
# Response as Pydantic object
print(response.parsed)

#### 💡 [OpenAPI Schema](https://swagger.io/specification/) support

Gemini 3.1 Pro offers native support for OpenAPI schemas. This allows engineers to define function calling interfaces using the industry-standard specification. It removes the need to write a custom boilerplate for every tool. It also ensures high compatibility with existing RESTful services and documentation 

In [ ]:
response_schema = {
    "required": [
        "name",
        "population",
        "capital",
        "continent",
        "gdp",
        "official_language",
        "total_area_sq_mi",
    ],
    "properties": {
        "name": {"type": "STRING"},
        "population": {"type": "INTEGER"},
        "capital": {"type": "STRING"},
        "continent": {"type": "STRING"},
        "gdp": {"type": "INTEGER"},
        "official_language": {"type": "STRING"},
        "total_area_sq_mi": {"type": "INTEGER"},
    },
    "type": "OBJECT",
}

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Give me information for the United States.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=response_schema,
    ),
)
# As JSON
display(Markdown(response.text))
# As Dict
print(response.parsed)

### ✅ Grounding with Google Search

Gemini 3.1 Pro has a search-integrated tool designed for complex engineering tasks. It is not a standard chatbot. It combines a 1-million-token context window with real-time reasoning to solve multi-step problems using text, images, and video.


In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Where will the next FIFA World Cup be held?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
    ),
)

display(Markdown(response.text))

grounding_metadata = response.candidates[0].grounding_metadata
if grounding_metadata and grounding_metadata.search_entry_point:
    display(HTML(grounding_metadata.search_entry_point.rendered_content))

Return to the lab instructions to **Check your progress** to verify you ran the **Grounding with Google Search** section.

### ✅ Code execution

The code execution feature allows the model to generate, run, and correct Python code in a secure environment. This allows the model to check its own logic, learning from the results until it reaches a final output.

In [ ]:
# Define code execution tool
code_execution_tool = types.Tool(code_execution=types.ToolCodeExecution())

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Calculate 20th fibonacci number. Then find the nearest palindrome to it.",
    config=types.GenerateContentConfig(
        tools=[code_execution_tool],
    ),
)

display(
    Markdown(
        f"""
## Code

```py
{response.executable_code}
```

### Output

```
{response.code_execution_result}
```
"""
    )
)

Return to the lab instructions to **Check your progress** to verify you ran the **Code execution** section.

### ✅ URL context

The Gemini 3.1 Pro URL context tool lets engineers include URLs as input to the model's reasoning engine. This allows live or indexed web content to be passed directly into the model. This eliminates the need to manually copy and paste data for analysis. 

In [ ]:
# 1. Define the tool correctly for Gemini
url_context_tool = types.Tool(url_context=types.UrlContext())

url = "https://blog.google/innovation-and-ai/models-and-research/gemini-models/gemini-3-1-pro/"

# 2. Call the model
# PRO-TIP: We use "Using the Browse tool" to ensure the agent triggers the URL fetch
response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Using the Browse tool, summarize this specific document: {url}",
    config=types.GenerateContentConfig(
        tools=[url_context_tool],
    ),
)

display(Markdown(response.text))

# 3. Robust Verification Logic
candidate = response.candidates[0]
print(f"--- Grounding & Tool Verification ---\n")

# Check URL Context Tool specific metadata
url_meta = getattr(candidate, 'url_context_metadata', None)
if url_meta and url_meta.url_metadata:
    print("✅ URL Context Tool was triggered:")
    for item in url_meta.url_metadata:
        print(f"   • Retrieved: {item.retrieved_url}")
        print(f"     Status: {item.url_retrieval_status}")

# Check standard Grounding Metadata (if search was also triggered)
grounding_meta = getattr(candidate, 'grounding_metadata', None)
if grounding_meta and grounding_meta.grounding_chunks:
    print("\n🔗 Sources cited in Grounding Metadata:")
    for i, chunk in enumerate(grounding_meta.grounding_chunks):
        title = chunk.web.title if chunk.web else "No Title"
        uri = chunk.web.uri if chunk.web else "No URI"
        print(f"   [{i}]: {title} ({uri})")
        
if not url_meta and not grounding_meta:
    print("⚠️ Note: The model answered from internal knowledge without using external tools.")

Return to the lab instructions to **Check your progress** to verify you ran the **URL context** section.

### ✅ Function calling

Function calling in Gemini 3.1 Pro enables the model to interact with external systems. It does this by outputting structured data instead of natural language. This feature is a core component of "agentic" workflows. It allows the model to perform real-world actions like querying databases, triggering API calls, or controlling local hardware. 


In [ ]:
def get_weather(location: str):
    """Get the current weather in a specific location.

    Args:
        location: The city and state, e.g. San Francisco, CA or a zip code.
    """
    # This is a placeholder for a real API call.
    return {"temperature": "32", "unit": "celsius"}


response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the weather like in Toronto?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
    ),
)

display(Markdown(response.text))

Return to the lab instructions to **Check your progress** to verify you ran the **Function calling** section.

### ✅ Streaming function calling

Gemini 3.1 Pro allows you to stream partial function call arguments in real time. This improves the streaming experience on tool use. To enable this feature, set `stream_function_call_arguments` to true.

In [ ]:
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Gets the current weather temperature for a given location.",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
)
get_weather_tool = types.Tool(function_declarations=[get_weather_declaration])


for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents="What's the weather in London and New York?",
    config=types.GenerateContentConfig(
        tools=[get_weather_tool],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode=types.FunctionCallingConfigMode.AUTO,
                stream_function_call_arguments=True,
            )
        ),
    ),
):
    if chunk.function_calls:
        function_call = chunk.function_calls[0]
        if function_call and function_call.name:
            print(f"{function_call.name}")
            print(f"will_continue={function_call.will_continue}")

### ✅ Multimodal function responses

Multimodal function responses allow agents to process visual or document-based outputs from external tools. This enables a bidirectional multimodal loop. 

In [ ]:
# 1. Define the function tool
get_image_declaration = types.FunctionDeclaration(
    name="get_image",
    description="Retrieves the image file reference for a specific order item.",
    parameters={
        "type": "object",
        "properties": {
            "item_name": {
                "type": "string",
                "description": "The name or description of the item ordered (e.g., 'green shirt').",
            }
        },
        "required": ["item_name"],
    },
)
tool_config = types.Tool(function_declarations=[get_image_declaration])

# 2. Send a message that triggers the tool
prompt = "Show me the green shirt I ordered last month."
response_1 = client.models.generate_content(
    model=MODEL_ID,
    contents=[prompt],
    config=types.GenerateContentConfig(
        tools=[tool_config],
    ),
)

# 3. Handle the function call
function_call = response_1.function_calls[0]
requested_item = function_call.args["item_name"]
print(f"Model wants to call: {function_call.name}")

# Execute your tool (e.g., call an API)
# (This is a mock response for the example)
print(f"Calling external tool for: {requested_item}")

function_response_data = {
    "image_url": "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/dress.jpg",
}

function_response_multimodal_data = types.FunctionResponsePart(
    file_data=types.FunctionResponseFileData(
        mime_type="image/jpeg",
        display_name="dress.jpg",
        file_uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/image/dress.jpg",
    )
)

# 4. Send the tool's result back
# Append this turn's messages to history for a final response.
history = [
    types.Content(role="user", parts=[types.Part(text=prompt)]),
    response_1.candidates[0].content,
    types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(
                name=function_call.name,
                response=function_response_data,
                parts=[function_response_multimodal_data],
            )
        ],
    ),
]

response_2 = client.models.generate_content(
    model=MODEL_ID,
    contents=history,
    config=types.GenerateContentConfig(
        tools=[tool_config],
        thinking_config=types.ThinkingConfig(include_thoughts=True),
    ),
)

print(f"\nFinal model response:")
display(Markdown(response_2.text))

### ✅ Thought signatures

[Thought signatures](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/thinking#signatures) are encrypted tokens that preserve the model's reasoning state during multi-turn conversations, specifically when using Function Calling.

When a thinking model decides to call an external tool, it pauses its internal reasoning process. The thought signature acts as a "save state", allowing the model to resume its chain of thought seamlessly once you provide the function's result.

Gemini 3.1 Pro enforces stricter validation and updated handling on thought signatures which were originally introduced in Gemini 2.5. To ensure the model maintains context across multiple turns of a conversation, you must return the thought signatures in your subsequent requests.


#### Automatic Handling of Thought Signatures (Recommended)

If you are using the Google Gen AI SDKs (Python, Node.js, Go, Java) or OpenAI Chat Completions API, and utilizing the standard chat history features or appending the full model response, thought signatures are handled automatically. You do not need to make any changes to your code.


#### **Example 1**: Automatic function calling

When using the Gen AI SDK in automatic function calling, thought signatures are handled automatically.


In [ ]:
def get_weather(city: str) -> str:
    """Gets the weather in a city."""
    if "london" in city.lower():
        return "Rainy"
    if "new york" in city.lower():
        return "Sunny"
    return "Cloudy"


response = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the weather in London and New York?",
    config=types.GenerateContentConfig(
        tools=[get_weather],
    ),
)

# The SDK handles the function calls and thought signatures, and returns the final text
print("Final response:", response.text)

# Print function calling history
hist_turn = response.automatic_function_calling_history[1]
print("\nFunction Call 1:", hist_turn.parts[1].function_call.name)

#### **Example 2**: Manual function calling

When using the Gen AI SDK in manual function calling, thought signatures are also handled automatically if you append the full model response in sequential model requests.

In [ ]:
# 1. Define your tool
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Gets the current weather temperature for a given location.",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
)
get_weather_tool = types.Tool(function_declarations=[get_weather_declaration])

# 2. Send a message that triggers the tool
prompt = "What's the weather like in London?"
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[get_weather_tool],
        thinking_config=types.ThinkingConfig(include_thoughts=True),
    ),
)

# 3. Handle the function call
function_call = response.function_calls[0]
location = function_call.args["location"]
print(f"Model wants to call: {function_call.name}")

# Execute your tool (e.g., call an API)
# (This is a mock response for the example)
print(f"Calling external tool for: {location}")
function_response_data = {
    "location": location,
    "temperature": "30C",
}

# 4. Send the tool's result back
# Append this turn's messages to history for a final response.
# The `content` object automatically attaches the required thought_signature behind the scenes.
history = [
    types.Content(role="user", parts=[types.Part(text=prompt)]),
    response.candidates[0].content,  # Signature preserved here
    types.Content(
        role="tool",
        parts=[
            types.Part.from_function_response(
                name=function_call.name,
                response=function_response_data,
            )
        ],
    ),
]

response_2 = client.models.generate_content(
    model=MODEL_ID,
    contents=history,
    config=types.GenerateContentConfig(
        tools=[get_weather_tool],
        thinking_config=types.ThinkingConfig(include_thoughts=True),
    ),
)

# 5. Get the final, natural-language answer
print(f"\nFinal model response: {response_2.text}")

#### Manual handling of thought signatures

If you are interacting with the API directly or managing raw JSON payloads, you must correctly handle the `thought_signature` included in the model's turn.
You must return this signature in the exact part where it was received when sending the conversation history back.

⚠️ If proper signatures are not returned, the model will return a 400 Error `"Function Call in the content block is missing a thought_signature"`.

See [documentation](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/multimodal/function-calling#thinking) for more details.

### ✅ Count tokens and compute tokens

Count tokens is a tool to manage costs and stay within the model's 1-million-token context window. This tool provides a precise count of the billing units used by different data types. These include text, code, high-resolution images, and video.

In [ ]:
# Count tokens
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="why is the sky blue?",
)

print(response)

In [ ]:
# Compute tokens
response = client.models.compute_tokens(
    model=MODEL_ID,
    contents="why is the sky blue?",
)

print(response)

Return to the lab instructions to **Check your progress** to verify you ran the **Count tokens and compute tokens** section of the lab.